In [1]:
import requests
import xml.etree.ElementTree as ET
import time
from urllib.parse import urlencode

In [2]:
# Variable declarations at start
BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
SEARCH_TERM = "Listeria AND english[lang]"
DATE_START = "2025/01/01"
DATE_END = "2025/06/30"
DATE_TYPE = "pdat"
FALLBACK_DATE_TYPE = "edat"
MAX_RESULTS = 10
RATE_LIMIT_DELAY = 0.34
DATABASE = "pubmed"
RETMODE_XML = "xml"
USE_HISTORY = "y"

In [3]:
def construct_esearch_url(database, term, mindate, maxdate, datetype, retmax, usehistory):
    params = {
        'db': database,
        'term': term,
        'mindate': mindate,
        'maxdate': maxdate,
        'datetype': datetype,
        'retmax': retmax,
        'usehistory': usehistory,
        'retmode': 'xml'
    }
    return BASE_URL + 'esearch.fcgi?' + urlencode(params)

def construct_efetch_url(database, pmids, retmode):
    id_string = ','.join(pmids)
    params = {
        'db': database,
        'id': id_string,
        'retmode': retmode,
    }
    return BASE_URL + 'efetch.fcgi?' + urlencode(params)

def make_http_request(url):
    response = requests.get(url)
    response.raise_for_status()
    return response.text

In [28]:
def extract_total_count(xml_response):
    root = ET.fromstring(xml_response)
    count_element = root.find('Count')

    if count_element is not None and count_element.text is not None:
        return int(count_element.text)
    return 0

def extract_pmid_list(xml_response):
    root = ET.fromstring(xml_response)
    pmid_list = []
    id_list = root.find('IdList')
    if id_list is not None:
        for id_element in id_list.findall('Id'):
          pmid_list.append(id_element.text)
    return pmid_list

def extract_webenv(xml_response):
    root = ET.fromstring(xml_response)
    webenv_element = root.find('WebEnv')
    if webenv_element is not None:
        return webenv_element.text
    return None

def extract_query_key(xml_response):
    root = ET.fromstring(xml_response)
    querykey_element = root.find('QueryKey')
    if querykey_element is not None:
        return querykey_element.text
    return None

def parse_articles_xml(xml_response):

    articles_data = []
    root = ET.fromstring(xml_response)

    for article in root.findall('.//PubmedArticle'):
        pmid_element = article.find('.//PMID')
        title_element = article.find('.//ArticleTitle')

        if pmid_element is not None:
          pmid = pmid_element.text
        else:
          pmid = "PMID not available"

        if title_element is not None:
          title = title_element.text
        else:
          title = "Title not available"

        #extract pmcid
        pmcid = "PMCID not available"
        article_id_list = article.find ('.//ArticleIdList')
        if article_id_list is not None:
            for article_id in article_id_list.findall('.//ArticleId'):
                id_type = article_id.get('IdType')
                if id_type == 'pmc':
                    pmcid = article_id.text
                    break

        #extract abstract
        abstract = "Abstract not available"
        abstract_element = article.find('.//Abstract')
        if abstract_element is not None:
            abstract_texts = []

            for abstract_text in abstract_element.findall('AbstractText'):
              label = abstract_text.get('Label', '')
              if abstract_text.text is not None:
                text = abstract_text.text
              else:
                text =''

              print(f"pmid: {pmid} abstract: {text}")
              if label and text:
                abstract_texts.append(f'{label}: {text}')
              else:
                abstract_texts.append(text)

            if abstract_texts:
                abstract = ' '.join(abstract_texts)

        articles_data.append({
            'pmid': pmid,
            'pmcid': pmcid,
            'title': title,
            'abstract': abstract
        })

    return articles_data

def search_with_fallback_date():
    print("Attempting search with fallback date type (Entrez date)...")

    global DATE_TYPE
    original_date_type = DATE_TYPE
    DATE_TYPE = FALLBACK_DATE_TYPE

    try:
        search_listeria_articles()
    finally:
        DATE_TYPE = original_date_type

In [13]:
def search_listeria_articles():
    print("=== Listeria PubMed Article Search ===")
    print(f"Search Term: {SEARCH_TERM}")
    print(f"Date Range: {DATE_START} to {DATE_END}")
    print(f"Max Results: {MAX_RESULTS}")
    print()

    # Step 1: Construct ESearch URL
    search_url = construct_esearch_url(
        database=DATABASE,
        term=SEARCH_TERM,
        mindate=DATE_START,
        maxdate=DATE_END,
        datetype=DATE_TYPE,
        retmax=MAX_RESULTS,
        usehistory=USE_HISTORY
    )

    print("Step 1: Searching PubMed...")

    # Step 2: Make API request with error handling
    try:
        response = make_http_request(search_url)
        time.sleep(RATE_LIMIT_DELAY)
    except requests.exceptions.RequestException as e:
        print(f"Error: API search request failed - {str(e)}")
        return None
    except Exception as e:
        print(f"Error: Unexpected error during search - {str(e)}")
        return None

    # Step 3: Parse XML response
    try:
        total_count = extract_total_count(response)
        pmid_list = extract_pmid_list(response)
        webenv = extract_webenv(response)
        query_key = extract_query_key(response)

        print(f"Total articles found: {total_count}")
        print(f"Retrieved PMIDs: {len(pmid_list)}")
        print()

    except ET.ParseError as e:
        print(f"Error: Failed to parse search response XML - {str(e)}")
        return None
    except Exception as e:
        print(f"Error: Failed to extract search data - {str(e)}")
        return None

    # Step 4: Fetch article details if PMIDs found
    if pmid_list:
        print("Step 2: Fetching article details...")

        fetch_url = construct_efetch_url(
            database=DATABASE,
            pmids=pmid_list,
            retmode=RETMODE_XML
        )

        try:
            articles_response = make_http_request(fetch_url)
            time.sleep(RATE_LIMIT_DELAY)
        except requests.exceptions.RequestException as e:
            print(f"Error: Failed to fetch article details - {str(e)}")
            return None
        except Exception as e:
            print(f"Error: Unexpected error during article fetch - {str(e)}")
            return None

        # Step 5: Parse articles and display results
        try:
            articles_data = parse_articles_xml(articles_response)

            print("=== RESULTS ===")
            if articles_data:
                for i, article in enumerate(articles_data, 1):
                    print(f"{i}. PMID: {article['pmid']}")
                    print(f"   PMCID: {article['pmcid']}")
                    print(f"   Title: {article['title']}")

                    #Handle abstract display
                    abstract_text = article['abstract']
                    if isinstance(abstract_text, str) and len(abstract_text) > 200:
                      print(f"   Abstract: {article['abstract'][:300]}..." )
                    else:
                      print(f"   Abstract: {abstract_text}")
            else:
                print("No article details could be parsed from the response")

        except ET.ParseError as e:
            print(f"Error: Failed to parse articles response XML - {str(e)}")
            return None
        except Exception as e:
            print(f"Error: Failed to parse article details - {str(e)}")
            return None
    else:
        print("No articles found for the specified criteria")
        print("You may want to:")
        print("1. Expand the date range")
        print("2. Try different search terms")
        print("3. Check if articles exist for this time period")

In [29]:
# Main execution
if __name__ == "__main__":
    print("Starting Listeria pathogen search in PubMed...")
    print("Note: This search covers Jan-Jun 2025 publications")
    print()

    # Primary search with publication date
    search_listeria_articles()

    #print("\n" + "="*50)
    #print("If no results were found, you can try:")
    #print("1. search_with_fallback_date() - Uses Entrez date instead of publication date")
    #print("2. Modify SEARCH_TERM variable and run search_listeria_articles() again")
    #print("3. Adjust DATE_START and DATE_END variables for different time ranges")

Starting Listeria pathogen search in PubMed...
Note: This search covers Jan-Jun 2025 publications

=== Listeria PubMed Article Search ===
Search Term: Listeria AND english[lang]
Date Range: 2025/01/01 to 2025/06/30
Max Results: 10

Step 1: Searching PubMed...
Total articles found: 651
Retrieved PMIDs: 10

Step 2: Fetching article details...
pmid: 40873491 abstract: AI-enabled microscopy is emerging for rapid bacterial classification, yet its utility remains limited in dynamic or resource-limited settings due to imaging variability. This study aims to enhance the generalizability of AI microscopy using domain adaptation techniques. Six bacterial species, including three Gram-positive (
pmid: 40831634 abstract: 
pmid: 40827030 abstract: The type I interferon family of cytokines are rapidly produced following innate pattern recognition receptor engagement and establish a critical early state of host defense. Type I interferons act in antiviral immunity as transcriptional activators and th